<a href="https://colab.research.google.com/github/gencere757/ARDA-GENCER-DSA210-TERM-PROJECT/blob/main/DSA210_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Filtering, Cleaning And Transforming The Data
## 1.1 Importing the necessary libraries


In [ ]:
import pandas as pd
import numpy as np

## 1.2 Obesity Rate Data
  Here, the obesity rate data were made into a dataframe, the empty or NaN values were cleared, and a seperate column for log values were added since they will be needed to apply log transform when plotting later.

In [ ]:
foodDF = pd.read_csv('StateAndCountyData.csv')
obesityDF = foodDF[foodDF['Variable_Code'] == "PCT_OBESE_ADULTS22"]
obesityDF = obesityDF.dropna()
obesityDF = obesityDF[obesityDF['Value']>0].copy()
obesityDF

## 1.3 The Walkability Index Data
Here, the walkability index data was extracted from its .csv file.

In [ ]:
walkDF = pd.read_csv('WalkabilityIndexByCounty.csv')
walkDF

## 1.4 Fast Food Restaurants Per 1000 People
The data was extracted from the .csv file, the empty rows were cleaned and log transformation was applied to data since the distribution here was mostly right-skewed.

In [ ]:
restaurantDF = foodDF[foodDF['Variable_Code'] == "FFRPTH20"]   #Fast food restaurants per 1000 people in county
restaurantDF = restaurantDF[restaurantDF['Value']>0].copy()
restaurantDF['Log_Transformed_Restaurants'] = np.log1p(restaurantDF['Value'])
restaurantDF

## 1.5 Combining  The Data Into A Single Data Frame

In [ ]:
walkSubset = walkDF[['GEOID10', 'NatWalkInd']].copy()
walkSubset = walkSubset.rename(columns={
    'GEOID10': 'FIPS',
    'NatWalkInd': 'Walkability_Index'
})

restSubset = restaurantDF[['FIPS', 'Value', 'Log_Transformed_Restaurants']].copy()
restSubset = restSubset.rename(columns={
    'Value': 'Fast_Food_Per_1000',
    'Log_Transformed_Restaurants': 'Log_Fast_Food'
})

obeseSubset = obesityDF[['FIPS', 'Value']].copy()
obeseSubset = obeseSubset.rename(columns={
    'Value': 'Obesity_Rate'
})

combinedDF = walkSubset.merge(obeseSubset, on='FIPS', how='inner') \
                       .merge(restSubset, on='FIPS', how='inner')
combinedDF = combinedDF.dropna()
combinedDF.to_csv('Processed Data.csv', index=False)
combinedDF

# 2. Data Visualizations

## 2.1 Graphs, Correlation, Preliminary Analysis

### 2.1.1 Importing The Necessary Libraries

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

### 2.1.1 Getting The General Statistical Information About The Data

In [3]:
combinedDF.describe()

NameError: name 'combinedDF' is not defined

### 2.1.2 Plotting The Walkability Index Data As A Histogram

To be able to have a general idea as to how the data are distributed, histogram plots were drawn for each of them.

In [ ]:
plt.figure(figsize=(9,6))
sns.histplot(combinedDF['Walkability_Index'],bins=50,kde=True,alpha=0.6)
plt.title("Distribution Of Walkability Index")
plt.xlabel("Walkability Index")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

### 2.1.3 Plotting The Fast Food Restaurant Data As A Histogram

In [ ]:
plt.figure(figsize=(9,6))
sns.histplot(combinedDF['Log_Fast_Food'],bins=50,kde=True,alpha=0.6)
plt.title("Distribution Of Fast Food Restaurants Per 1000 Population")
plt.xlabel("Number of Fast Food Restaurants Per 1000 Population (Log Transformed)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

### 2.1.2 Plotting The Obesity Rate Data As A Histogram



In [ ]:
plt.figure(figsize=(9,6))
sns.histplot(combinedDF['Obesity_Rate'],bins=20,kde=True,alpha=0.6)
plt.title("Distribution Of Obesity Rate")
plt.xlabel("Obesity Rate")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

### 2.1.3 Mapping The Walkability Index And Fast Food Data Against Obesity Rates As A Scatterplot

The scatterplots were used as to see if there is any correlation that can be seen by the eye between the features and the obesity rate.

In [ ]:
sns.scatterplot(x=combinedDF['Walkability_Index'], y=combinedDF['Obesity_Rate'],alpha=0.25)
plt.title('Obesity Rate vs Walkability Index')
plt.xlabel('Walkability Index')
plt.ylabel('Log-transformed Mortality Rate')
plt.tight_layout()
plt.show()

sns.scatterplot(x=combinedDF['Log_Fast_Food'], y=combinedDF['Obesity_Rate'],alpha=0.25)
plt.title('Obesity Rate vs Number Of Fast Food Restaurants Per 1000 People')
plt.xlabel('Number of Fast Food Restaurants Per 1000 Population (Log Transformed)')
plt.ylabel('Log-transformed Mortality Rate')
plt.tight_layout()
plt.show()

### 2.1.4 Creating A Correlation Matrix To Compare Each Data Against One Another

In [ ]:
plt.figure(figsize=(9, 8))
sns.heatmap(combinedDF[['Walkability_Index', 'Log_Fast_Food', 'Obesity_Rate']].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')

As can be seen, we see almost no correlation between fast food restaurant count and obesity rate. However, between walkabilty index and obesity rate, a quite large negative correlation can be observed.

## 2.2 Mapping The Data

Since the data that will be used dependds on locations (i.e counties in US), the data were plotted on a map for better visualization.

### 2.2.1 Importing The Necessary Libraries To Map The Data

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

### 2.2.2 Mapping The Obesity Rate Data On The US Map

In [ ]:
combinedDF['FIPS'] = combinedDF['FIPS'].astype(str).str.zfill(5)

fig = px.choropleth(
    combinedDF,
    geojson="https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json",
    locations='FIPS',
    color='Obesity_Rate',
    color_continuous_scale="Reds",
    scope="usa",
    hover_name='FIPS',
    hover_data={
        'FIPS': False,
        'Obesity_Rate': ':.1f',
        'Walkability_Index': ':.2f',
        'Fast_Food_Per_1000': ':.3f'
    },
    labels={'Obesity_Rate': 'Obesity_Rate'},
    title='US County-Level Data Visualization For Obesity Rates'
)

fig.update_layout(
    geo=dict(
        lakecolor='rgb(255, 255, 255)',
    ),
    margin={"r": 0, "t": 50, "l": 0, "b": 0},
    height=600
)

fig.show()

### 2.2.3 Mapping The Walkability Index Data On The US Map

In [ ]:
combinedDF['FIPS'] = combinedDF['FIPS'].astype(str).str.zfill(5)

fig = px.choropleth(
    combinedDF,
    geojson="https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json",
    locations='FIPS',
    color='Walkability_Index',
    color_continuous_scale="Reds",
    scope="usa",
    hover_name='FIPS',
    hover_data={
        'FIPS': False,
        'Obesity_Rate': ':.1f',
        'Walkability_Index': ':.2f',
        'Fast_Food_Per_1000': ':.3f'
    },
    labels={'Walkability_Index': 'Walkability_Index'},
    title='US County-Level Data Visualization For Walkability Index'
)

fig.update_layout(
    geo=dict(
        lakecolor='rgb(255, 255, 255)',
    ),
    margin={"r": 0, "t": 50, "l": 0, "b": 0},
    height=600
)

fig.show()

### 2.2.4 Mapping The Fast Food Restaurant Data On The US Map

In [ ]:
combinedDF['FIPS'] = combinedDF['FIPS'].astype(str).str.zfill(5)

fig = px.choropleth(
    combinedDF,
    geojson="https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json",
    locations='FIPS',
    color='Log_Fast_Food',
    color_continuous_scale="Reds",
    scope="usa",
    hover_name='FIPS',
    hover_data={
        'FIPS': False,
        'Obesity_Rate': ':.1f',
        'Walkability_Index': ':.2f',
        'Fast_Food_Per_1000': ':.3f'
    },
    labels={'Log_Fast_Food': 'Log_Fast_Food'},
    title='US County-Level Data Visualization For Fast Food Restaurant Per 1000 Population'
)

fig.update_layout(
    geo=dict(
        lakecolor='rgb(255, 255, 255)',
    ),
    margin={"r": 0, "t": 50, "l": 0, "b": 0},
    height=600
)

fig.show()

# 3. Hypothesis Testing



## 3.1 Defining Hypotheses And Testing Methods

Our Hypotheses are as follows:

1.   Walkability Index

Hₒ : The walkability index and obesity rate is not negatively correlated.

Hₐ : The walkability index and obesity rate is negatively correlated. As walkability index gets higher, obesity rates get lower.

2.   Fast Food Data

Hₒ : The fast food restaurant count per 1000 people and obesity rate is not positively correlated.

Hₐ : The fast food restaurant count per 1000 people and obesity rate is positively correlated.




---


Here, permutation testing was used to calculate different distributions assuming that the null hypotheses were true. After that, the correlation coefficients for these were saved and the real correlation coefficient (here, spearman correlation was used as obesity data didn't seem normally distributed and pearson correlation is a parametric test) was compared against this distribution of correlation coefficients to determine if there was any correlation.

In [ ]:
from scipy import stats

## 3.2 Permutation Test Function

In [ ]:
def permutation_test_correlation(x, y, n_permutations=10000, alternative='two-sided'):
    x = np.asarray(x)
    y = np.asarray(y)

    observed_corr = stats.spearmanr(x, y).correlation

    perm_corrs = np.array([
        stats.spearmanr(x, np.random.permutation(y)).correlation
        for _ in range(n_permutations)
    ])

    if alternative == 'two-sided':
        extreme = np.sum(np.abs(perm_corrs) >= np.abs(observed_corr))
    elif alternative == 'less':      # H1: observed_corr is smaller (more negative)
        extreme = np.sum(perm_corrs <= observed_corr)
    elif alternative == 'greater':   # H1: observed_corr is larger (more positive)
        extreme = np.sum(perm_corrs >= observed_corr)
    else:
        raise ValueError("alternative must be 'two-sided', 'less', or 'greater'")

    # Monte Carlo p-value correction
    p_value = (extreme + 1) / (n_permutations + 1)
    return observed_corr, p_value, perm_corrs

In [ ]:
#Walkabilty
obs_corr_walkability, p_val_walkability, perm_dist_walkability = permutation_test_correlation(
    combinedDF['Walkability_Index'],
    combinedDF['Obesity_Rate'],
    n_permutations=10000,
    alternative='less'  # H1: negative correlation
)
print(f"Walkability Index\n")
print(f"Observed correlation: {obs_corr_walkability:.4f}")
print(f"P-value (one-sided, H1: r < 0): {p_val_walkability:.4f}")
print(f"Interpretation: {'Significant' if p_val_walkability < 0.05 else 'Not significant'} at α=0.05")
if p_val_walkability < 0.05:
  print(f"The walkability index is negatively correlated with obesity rate."),
else:
  print(f"The walkability index is not negatively correlated with obesity rate.")
print('\n')

#Fast Food Data
obs_corr_food, p_val_food, perm_dist_food = permutation_test_correlation(
    combinedDF['Log_Fast_Food'],
    combinedDF['Obesity_Rate'],
    n_permutations=10000,
    alternative='greater'  # H1: positive correlation
)
print(f"Fast Food Data\n")
print(f"Observed correlation: {obs_corr_food:.4f}")
print(f"P-value (one-sided, H1: r < 0): {p_val_food:.4f}")
print(f"Interpretation: {'Significant' if p_val_food < 0.05 else 'Not significant'} at α=0.05")
if p_val_food < 0.05:
  print(f"The fast food restaurant count per 1000 people is positively correlated with the obesity rate.")
else:
  print(f"The fast food restaurant count per 1000 people is not positively correlated with the obesity rate.")

## 3.2 Plotting The Distributions And Observed Correlations

Here, to visualize  it better, the randomly generated distributions and the observed correlation is plotted in the same graph.

In [ ]:
plt.figure(figsize=(9,6))
sns.histplot(perm_dist_walkability,bins=100,kde=True,alpha=0.6)
plt.title("Permutation Test - Walkability Index vs. Obesity Rate")
plt.axvline(x=obs_corr_walkability, color='red', linestyle='--', label = 'Observed Correlation')
plt.xlabel("Correlation Coefficient")
plt.ylabel("Frequency")
plt.tight_layout()
plt.legend()
plt.show()

plt.figure(figsize=(9,6))
sns.histplot(perm_dist_food,bins=100,kde=True,alpha=0.6)
plt.title("Permutation Test - Walkability Index vs. Obesity Rate")
plt.axvline(x=obs_corr_food, color='red', linestyle='--', label = 'Observed Correlation')
plt.xlabel("Correlation Coefficient")
plt.ylabel("Frequency")
plt.tight_layout()
plt.legend()
plt.show()

# 4. Machine Learning


In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
combinedDF

In [ ]:
predictors = ["Walkability_Index","Log_Fast_Food"]
outcome = "Obesity_Rate"

In [ ]:
VIF_calc_X = combinedDF[predictors]
VIF_calc_X = sm.add_constant(VIF_calc_X)

VIF_data = pd.DataFrame()
VIF_data["feature"] = VIF_calc_X.columns
VIF_data["VIF"] = [variance_inflation_factor(VIF_calc_X.values,i) for i in range(VIF_calc_X.shape[1])]

VIF_data

In [ ]:
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

In [ ]:
X = combinedDF[predictors].values
y = combinedDF[outcome].values

# Defining Models and Hyperparameter Grids
models = {
    "LinearRegression": (
        Pipeline([
            ('scaler', StandardScaler()),
            ('model', LinearRegression())
        ]),
        {}
    ),
    "KNN": (
        Pipeline([
            ('scaler', StandardScaler()),
            ('model', KNeighborsRegressor())
        ]),
        {
            'model__n_neighbors': list(range(1,21))
        }
    ),
    "DecisionTree": (
        Pipeline([
            ('scaler', StandardScaler()),
            ('model', DecisionTreeRegressor())
        ]),
        {
            'model__max_depth': [3, 5, 10, 20, 30, None],
            'model__min_samples_leaf' : [2, 5, 10]
        }
    ),
    "RandomForest": (
        Pipeline([
            ('scaler', StandardScaler()),
            ('model', RandomForestRegressor())
        ]),
        {
            'model__n_estimators': [5, 10, 50],
            'model__max_depth': [5, 10, None]
        }
    )
}

outer_cv = KFold(n_splits=10, shuffle=True, random_state=42)
results = {}

In [ ]:
for name, (pipe, param_grid) in models.items():
    y_preds = np.zeros_like(y)
    best_params = []

    for train_idx, test_idx in outer_cv.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Inner CV with grid search
        inner_cv = KFold(n_splits=10, shuffle=True, random_state=1)
        grid = GridSearchCV(pipe, param_grid, cv=inner_cv, scoring='r2', n_jobs=-1)
        grid.fit(X_train, y_train)
        best_params.append(grid.best_params_)

        # Predict on outer test fold
        y_preds[test_idx] = grid.predict(X_test)

    results[name] = {"predictions": y_preds, "best_params": best_params}
    print(f"{name} R²: {r2_score(y, y_preds):.3f}, RMSE: {np.sqrt(np.mean((y - y_preds)**2)):.3f}")



In [ ]:
print("\nBest hyperparameters per model (one per outer fold):")
for name, result in results.items():
  print(f"\n{name}:")
  for i, params in enumerate(result['best_params']):
      print(f"  Fold {i + 1}: {params}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, (name, result) in enumerate(results.items()):
  ax = axes[i]
  y_pred = result["predictions"]
  ax.grid(True)
  ax.set_xticks(np.arange(4.6, 6.2, 0.2))
  ax.set_yticks(np.arange(4.6, 6.2, 0.2))
  min_val = min(y.min(), y_pred.min())
  max_val = max(y.max(), y_pred.max())
  ax.plot([min_val, max_val], [min_val, max_val], 'r--', label="Ideal Fit")
  ax.scatter(y, y_pred, alpha=0.3, edgecolor="none", s=30, label=name)
  ax.plot([4.6,6.2], [4.6,6.2], 'r--', label = "Ideal Fit")  # y = x line
  ax.set_title(f"{name}: Actual and Predicted (R² = {r2_score(y, y_pred):.2f} and RMSE = {np.sqrt(np.mean((y - y_preds)**2)):.2f})")
  ax.set_xlabel("Actual")
  ax.set_ylabel("Predicted")
  ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
  padding = 0.5
  min_val = min(y.min(), y_pred.min()) - padding
  max_val = max(y.max(), y_pred.max()) + padding
  ax.set_xlim(min_val, max_val)
  ax.set_ylim(min_val, max_val)

plt.tight_layout()
plt.show()

plt.tight_layout()
plt.show()

In [1]:
coefs = lr_pipe.named_steps["model"].coef_
for name, coef in zip(predictors, coefs):
    print(f"{name}: {coef:.4f}")
print("Linear Regression Intercept:", lr_pipe.named_steps['model'].intercept_)

NameError: name 'lr_pipe' is not defined